# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# 1. Method Choice and Why

For this capstone, I selected **Random Forest Classifier**.

My lane is **Content Opportunity & Search Growth**, and the target is **trend_direction** (up, down, stable).

Random Forest is suitable because it can learn nonlinear relationships between search-performance features, handle both numerical and categorical variables after encoding, and provide feature importance for interpretation. It is also a strong improvement over a simple baseline while remaining easy to explain.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

# 2. Split Design

I used the same train/test split as my Week 4 baseline to ensure a fair comparison.

- Train: 80%
- Test: 20%
- Random State: 42

Using the same split means any performance difference comes from the model rather than different data partitions. This provides an honest comparison.

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("content_refresh_anonymized.csv")

target = "trend_direction"

drop_cols = [
    "content_id",
    "client_id",
    "trend_pct"
]

X = df.drop(columns=drop_cols + [target])
y = df[target]

categorical = X.select_dtypes(include="object").columns
numeric = X.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", "passthrough", numeric)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [16]:
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)

rf_accuracy = accuracy_score(y_test, pred)

print("Random Forest Accuracy:", rf_accuracy)

print(classification_report(y_test, pred))

Random Forest Accuracy: 0.7708333333333334
              precision    recall  f1-score   support

        down       0.76      0.95      0.85      3252
        flat       1.00      1.00      1.00       231
         new       1.00      1.00      1.00       447
      stable       0.58      0.37      0.45      1192
          up       0.81      0.47      0.60       878

    accuracy                           0.77      6000
   macro avg       0.83      0.76      0.78      6000
weighted avg       0.76      0.77      0.75      6000



In [17]:
comparison = pd.DataFrame({
    "Model": [
        "Week 4 Baseline",
        "Random Forest"
    ],
    "Accuracy": [
        0.7312,
        round(rf_accuracy, 4)
    ]
})

comparison

,Model,Accuracy
0,Week 4 Baseline,0.7312
1,Random Forest,0.7708


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

###  Errors and Interpretation

The Random Forest model achieved an accuracy of **77.08%**, improving over the Week 4 baseline accuracy of **73.12%** on the same train/test split.

Most prediction errors occurred between the **stable** and **down** classes, suggesting these classes have overlapping search-performance characteristics.

The model relies on a combination of search volume, CTR, engagement, average position, and other content-related features rather than a single variable.

These results should be interpreted as decision-support rather than perfect predictions.

#Feature Importance

In [18]:
feature_names = model.named_steps["prep"].get_feature_names_out()

importance = model.named_steps["rf"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

importance_df.sort_values(
    by="Importance",
    ascending=False
).head(15)

,Feature,Importance
66,num__impressions_prev_30d,0.151566
63,num__impressions_last_30d,0.127897
53,num__impressions_90d,0.063778
73,num__avg_position,0.056944
61,num__days_with_impressions,0.049983
69,num__content_age_days,0.033313
65,num__sessions_last_30d,0.027446
51,num__word_count,0.027122
52,num__char_count,0.026592
72,num__ctr,0.023410


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.